# Simulated US Transactions for Credit Customers

Generates 6 months of USD, US-bank transactions for every customer in
`data/cleaned_credit/cleaned_train.csv`, with money-laundering typologies injected
so the `Is Laundering` label is **learnable**.


## Design


**Label.** A transaction is `Is Laundering = 1` **if and only if** it is part of an injected
typology. Nothing else sets it. The features that make a typology detectable — burst timing,
sub-\$10k amounts, counterparty concentration, cycle membership — are the same features that
create the label, which is what makes the problem learnable.

**Volume from account count, not income.** The Fed's 2024 Diary of Consumer Payment Choice
puts US consumers at ~48 payments/month. That is per *consumer*, not per account: someone
with six accounts does not spend six times as much. So external payment volume is drawn
independently of account count, and `Num_Bank_Accounts` drives **internal transfers between
the customer's own accounts**, which is what genuinely scales. This is also what produces
recurring counterparties, and therefore graph structure.

**Amounts and formats fitted from the real IBM data**, USD and US-banks only, conditional on
payment format. Measured on `LI-Small` (US-to-US, USD): Credit Card is tight (std_log 2.18,
11% ≥ \$10k) while Reinvestment is heavy-tailed (std_log 3.98, 31.6% ≥ \$10k). Conditioning
amount on format is what puts realistic mass in the \$10,000 region where BSA rules live.

Entity-type conditioning was dropped deliberately: Sole Proprietorship, Partnership and
Corporation are statistically identical in this data (mean_log 6.911 / 6.930 / 6.927), and
`Individual` has only 563 qualifying transactions and zero laundering examples across all of
LI-Small — too thin to fit anything on.

**Prevalence is per customer, at 3%.** 3% of 12,500 = 375 laundering customers, ~30 flagged
transactions each, giving ~0.1% at transaction level — within a factor of two of the real
LI-Small rate (0.0515%) while leaving enough positives to train on. 3% of customers
laundering is far above any real portfolio; it buys trainability, and it means precision and
recall from this data will not transfer to a real book without recalibrating against true
prevalence.

**Reproducibility.** All randomness comes from seeded `np.random.default_rng`, and every ID
hash uses `hashlib.md5` — never Python's `hash()`.

Sources: [2024 Diary of Consumer Payment Choice](https://www.frbservices.org/news/research/2024-findings-from-the-diary-of-consumer-payment-choice) ·
[FinCEN CTR/SAR thresholds](https://bsaaml.ffiec.gov/manual/AssessingComplianceWithBSARegulatoryRequirements/05)

In [1]:
import hashlib
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# --- paths ---
REPO = Path.cwd().parent if Path.cwd().name == "EDA" else Path.cwd()
CREDIT_CSV   = REPO / "data" / "cleaned_credit" / "cleaned_train.csv"
REF_TRANS    = REPO / "data" / "aml" / "LI-Small_Trans.csv"
REF_ACCOUNTS = REPO / "data" / "aml" / "LI-Small_accounts.csv"
OUT_DIR      = REPO / "data" / "simulated"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- simulation window -------------------------------------------------------
MONTHS          = 6
SIMULATION_DAYS = 182
START_DATE      = pd.Timestamp("2024-01-01")

# --- volume ------------------------------------------------------------------
# 2024 Diary of Consumer Payment Choice: ~48 payments/month per consumer. Applied
# per CUSTOMER, not per account ->  more accounts does not mean more spending.
EXT_PAYMENTS_PER_MONTH   = 47.0
# Internal transfers DO scale with account count: each adjacent pair of the
# customer's own accounts becomes a recurring transfer route.
TRANSFERS_PER_ROUTE_MONTH = 3.0

# --- occupation tiers --------------------------------------------------------
# IMPORTANT, PLEASE READ BEFORE CITING THESE.
#
# No published source gives payment COUNT by occupation. The Fed's Diary of
# Consumer Payment Choice -- the source of the 47/month baseline above -- breaks
# payments down by income, age, education and instrument, but not by job. So the
# multipliers below are REASONED ASSUMPTIONS, not measurements, and each is
# recorded with the mechanism it is meant to represent:
#
#   Entrepreneur 1.60  business and personal flows run through one set of
#                      accounts, so the same person generates two streams. This
#                      is the only tier with a genuine structural mechanism, and
#                      the notebook already treats entrepreneurs as Sole
#                      Proprietorships for entity type.
#   Gig/irregular 1.20 income arrives from many payers in smaller amounts rather
#                      than one salary, raising inbound count (Musician, Writer,
#                      Journalist).
#   Cash trade    1.15 frequent small-value transactions (Mechanic).
#   Salaried      0.95 one regular salary, household spending, fewer counterparties.
#
# A caveat specific to this dataset: Occupation in cleaned_train.csv is assigned
# independently of income -- median income runs 34.7k to 39.5k across all fifteen
# jobs, with doctors below mechanics. So these tiers impose a structure that is
# NOT present in the source data. That is a deliberate modelling choice, and it
# means the AML model can partially infer occupation from transaction volume.
OCCUPATION_RATE_MULTIPLIER = {
    "Entrepreneur": 1.60,
    "Musician": 1.20, "Writer": 1.20, "Journalist": 1.20,
    "Mechanic": 1.15,
    "Doctor": 0.95, "Lawyer": 0.95, "Engineer": 0.95, "Architect": 0.95,
    "Accountant": 0.95, "Scientist": 0.95, "Developer": 0.95, "Teacher": 0.95,
    "Manager": 0.95, "Media_Manager": 0.95,
}
DEFAULT_RATE_MULTIPLIER = 1.00   # Unknown / unseen occupations

# --- per-customer heterogeneity ----------------------------------------------
# Plain Poisson forces variance == mean, which made every customer within an
# account tier nearly identical: measured spread on the previous output was only
# about +/-5%. Real payment counts are heavily overdispersed, and an
# unrealistically tight baseline makes laundering bursts stand out more sharply
# than they would in production -- it flatters the detector.
#
# Drawing each customer's own rate from a Gamma and then sampling Poisson given
# that rate is a negative binomial marginally, which decouples variance from
# mean. shape=4 gives a coefficient of variation of 50%, so a p90 customer
# transacts roughly 3x a p10 customer. Lower shape = more spread.
GAMMA_SHAPE = 4.0

# --- accounts ----------------------------------------------------------------
MIN_ACCOUNTS, MAX_ACCOUNTS = 1, 12   # 0 is a data artefact; clipped and flagged

# --- counterparties ----------------------------------------------------------
N_MERCHANT_ACCOUNTS = 40_000         # shared external pool (merchants, billers)
MERCHANTS_PER_CUSTOMER = (5, 15)
PEERS_PER_CUSTOMER     = (3, 8)
ZIPF_A = 1.3                          # counterparty reuse skew: a few dominate

# --- laundering --------------------------------------------------------------
LAUNDERING_CUSTOMER_RATE = 0.03      # per CUSTOMER, see markdown above
TYPOLOGIES = ["STRUCTURING", "FAN_IN", "FAN_OUT", "SCATTER_GATHER", "CYCLE"]
TYPOLOGY_WEIGHTS = [0.30, 0.20, 0.20, 0.15, 0.15]

# Structuring sits just under the $10,000 CTR threshold on purpose -- that pile-up
# is the detectable signature, and it is absent from the old data.
#
# Amounts are drawn as STRUCTURING_MAX - Exponential(scale), not uniformly across
# the band. Someone structuring wants the fewest possible trips while staying under
# reporting, so deposits cluster just below the threshold and thin out downward.
# A uniform draw spreads mass evenly from $2,000 and produces no pile-up at all --
# which is exactly what the validation cell caught on the first run.
STRUCTURING_MIN, STRUCTURING_MAX = 2_000, 9_900
STRUCTURING_DECAY = 1_500

SEED = 20240101
rng = np.random.default_rng(SEED)

def stable_bucket(key: str, mod: int = 100) -> int:
    """Deterministic bucket from a string. md5, not hash() -- Python randomises
    string hashing per process, which made the old notebook irreproducible."""
    return int(hashlib.md5(str(key).encode("utf-8")).hexdigest(), 16) % mod


# --- progress reporting ------------------------------------------------------
def fmt_secs(s: float) -> str:
    s = int(s)
    return f"{s//60}m{s%60:02d}s" if s >= 60 else f"{s}s"

def progress(done: int, total: int, t0: float, label: str, every: int = 1) -> None:
    """Print percent complete with elapsed time and an ETA extrapolated from the
    rate so far. Call once per iteration; it self-throttles via `every`."""
    if done % every and done != total:
        return
    elapsed = time.time() - t0
    rate = done / elapsed if elapsed > 0 else 0
    remaining = (total - done) / rate if rate > 0 else 0
    print(f"  {label}: {done:,}/{total:,} ({100*done/total:5.1f}%)  "
          f"elapsed {fmt_secs(elapsed)}  eta {fmt_secs(remaining)}", flush=True)

def stage(msg: str, t0: float = None) -> float:
    """Mark the start or end of a non-looping stage that takes real time."""
    if t0 is None:
        print(f"  {msg} ...", flush=True)
        return time.time()
    print(f"  {msg} done in {fmt_secs(time.time() - t0)}", flush=True)
    return time.time()


print(f"repo         : {REPO}")
print(f"window       : {MONTHS} months / {SIMULATION_DAYS} days from {START_DATE.date()}")
print(f"credit csv   : {CREDIT_CSV.exists()}")
print(f"reference    : {REF_TRANS.exists()} / {REF_ACCOUNTS.exists()}")

repo         : C:\dev\FRAML_PROJECT
window       : 6 months / 182 days from 2024-01-01
credit csv   : True
reference    : True / True


## Step 1: Fit amount and format models from the real IBM data

Restricted to **USD** transactions between two **US banks**, which is the population being
simulated. Amount parameters are fitted per payment format; the laundering-conditional format
mix is captured separately, because in the real data ACH accounts for 78% of laundering
transactions against 11.9% overall. That is a genuine signal, and reproducing it means the
injected typologies carry a pattern that can be justified from source data rather than invented.

Streams the 621 MB file in chunks — nothing large is held in memory.

In [2]:
def fit_reference_models():
    acc = pd.read_csv(REF_ACCOUNTS, usecols=["Bank Name", "Bank ID"],
                      dtype={"Bank Name": "string", "Bank ID": "int32"})
    # Foreign banks in this dataset are named "<Country> Bank #<n>"; everything
    # else ("National Bank of Albany", "Spruce Credit Union", ...) is domestic.
    foreign = acc["Bank Name"].str.match(r"^[A-Za-z ]+ Bank #\d+$")
    us_bank_ids = np.sort(acc.loc[~foreign, "Bank ID"].unique())
    us_set = set(us_bank_ids.tolist())

    n = {}; s1 = {}; s2 = {}
    fmt_all = pd.Series(dtype=float)
    fmt_laundering = pd.Series(dtype=float)

    cols = ["From Bank", "To Bank", "Amount Received", "Receiving Currency",
            "Payment Format", "Is Laundering"]
    # ~5 chunks across 6.9M rows. This stage is disk-bound on a 621 MB file, so
    # it can sit at low CPU for a while -- printing each chunk distinguishes
    # "reading slowly" from "stopped".
    print(f"  reading {REF_TRANS.name} (621 MB) in 1.5M-row chunks", flush=True)
    t_fit = time.time()
    n_chunks = 0
    for chunk in pd.read_csv(REF_TRANS, usecols=cols, chunksize=1_500_000,
                             dtype={"From Bank": "int32", "To Bank": "int32",
                                    "Amount Received": "float64", "Receiving Currency": "string",
                                    "Payment Format": "string", "Is Laundering": "int8"}):
        n_chunks += 1
        print(f"    chunk {n_chunks} read  (elapsed {fmt_secs(time.time() - t_fit)})", flush=True)
        chunk = chunk[(chunk["Receiving Currency"] == "US Dollar")
                      & chunk["From Bank"].isin(us_set)
                      & chunk["To Bank"].isin(us_set)
                      & (chunk["Amount Received"] > 0)]
        if chunk.empty:
            continue
        fmt_all = fmt_all.add(chunk["Payment Format"].value_counts(), fill_value=0)
        laundering = chunk[chunk["Is Laundering"] == 1]
        if len(laundering):
            fmt_laundering = fmt_laundering.add(
                laundering["Payment Format"].value_counts(), fill_value=0)
        for fmt, grp in chunk.groupby("Payment Format", observed=True):
            log_amt = np.log(grp["Amount Received"].to_numpy())
            n[fmt] = n.get(fmt, 0) + len(log_amt)
            s1[fmt] = s1.get(fmt, 0.0) + log_amt.sum()
            s2[fmt] = s2.get(fmt, 0.0) + (log_amt ** 2).sum()

    amount_models = {f: {"mean_log": s1[f] / n[f],
                         "std_log": float(np.sqrt(max(s2[f] / n[f] - (s1[f] / n[f]) ** 2, 1e-9))),
                         "n": n[f]}
                     for f in n}

    # Bitcoin is a rounding error in the US/USD slice; dropping it avoids a
    # category that appears a handful of times and destabilises encoding.
    fmt_all = fmt_all[fmt_all > fmt_all.sum() * 1e-4]
    fmt_all = fmt_all / fmt_all.sum()
    fmt_laundering = fmt_laundering.reindex(fmt_all.index).fillna(0.0)
    if fmt_laundering.sum() == 0:
        fmt_laundering = fmt_all.copy()
    fmt_laundering = fmt_laundering / fmt_laundering.sum()

    return us_bank_ids, amount_models, fmt_all, fmt_laundering


US_BANK_IDS, AMOUNT_MODELS, FORMAT_P, FORMAT_P_LAUNDERING = fit_reference_models()
FORMATS = list(FORMAT_P.index)

print(f"US bank IDs available : {len(US_BANK_IDS):,}\n")
print(f"{'format':<14}{'overall%':>10}{'laundering%':>13}{'mean_log':>10}{'std_log':>9}{'median $':>11}")
for f in FORMATS:
    m = AMOUNT_MODELS[f]
    print(f"{f:<14}{100*FORMAT_P[f]:>10.2f}{100*FORMAT_P_LAUNDERING[f]:>13.2f}"
          f"{m['mean_log']:>10.4f}{m['std_log']:>9.4f}{np.exp(m['mean_log']):>11,.0f}")

  reading LI-Small_Trans.csv (621 MB) in 1.5M-row chunks
    chunk 1 read  (elapsed 1s)
    chunk 2 read  (elapsed 3s)
    chunk 3 read  (elapsed 5s)
    chunk 4 read  (elapsed 7s)
    chunk 5 read  (elapsed 8s)
US bank IDs available : 14,631

format          overall%  laundering%  mean_log  std_log   median $
ACH                10.76        74.11    7.0394   3.0536      1,141
Cash                9.98         3.64    7.2165   3.1961      1,362
Cheque             38.16        14.16    7.2920   2.6285      1,468
Credit Card        27.38         8.09    6.3620   2.1833        579
Reinvestment       10.28         0.00    6.7672   3.9821        869
Wire                3.44         0.00    6.5143   3.6208        675


## Step 2: Customer profiles

One row per customer from `cleaned_train.csv`. Income is **not** read: it played no part in
this design. Deriving amounts from income and then validating income against those
amounts would be circular.

`Num_Bank_Accounts` is aggregated by maximum value across the customer's monthly rows and clipped to
[1, 12]. 546 customers carry 0 accounts, which is impossible for someone holding credit
products — it is a known-noisy column in this dataset. They are clipped up to 1 rather than
resampled, so no data is invented, and `Accounts_Imputed` marks them so the resulting spike at
1 account stays traceable.

In [3]:
credit = pd.read_csv(CREDIT_CSV, usecols=["Customer_ID", "Num_Bank_Accounts", "Occupation"],
                     low_memory=False)

profile = (credit.groupby("Customer_ID")
           .agg(Num_Bank_Accounts=("Num_Bank_Accounts", "max"),
                Occupation=("Occupation", lambda s: s.mode().iloc[0] if not s.mode().empty else "Unknown"))
           .reset_index())

raw_accounts = profile["Num_Bank_Accounts"].round()
profile["Accounts_Imputed"] = raw_accounts < MIN_ACCOUNTS
profile["Num_Bank_Accounts"] = raw_accounts.clip(MIN_ACCOUNTS, MAX_ACCOUNTS).astype(int)

# Occupation is retained for reporting only. It is NOT used to pick an amount
# model: the entity types in the reference data are statistically identical, and
# the one retail-looking type ("Individual") has too few observations to fit.
profile["Entity_Type"] = np.where(profile["Occupation"] == "Entrepreneur",
                                  "Sole Proprietorship", "Individual")

# Same MD5 customer-level partition as ml_models/data_splits.py, so a customer in
# CDA's test split is in AML's test split. If the two used different partitions the
# system slice would be training data for one of the models.
SPLIT_BOUNDS = [("train", 60), ("val", 75), ("test", 90), ("system", 100)]
def assign_split(cid):
    b = stable_bucket(cid)
    for name, upper in SPLIT_BOUNDS:
        if b < upper:
            return name
    return "system"

profile["Split"] = profile["Customer_ID"].map(assign_split)

N_CUSTOMERS = len(profile)
print(f"customers            : {N_CUSTOMERS:,}")
print(f"accounts imputed (0->1): {int(profile['Accounts_Imputed'].sum()):,}")
print(f"\naccounts per customer:\n{profile['Num_Bank_Accounts'].describe().to_string()}")
print(f"\nsplit sizes:\n{profile['Split'].value_counts().reindex([s for s,_ in SPLIT_BOUNDS]).to_string()}")

customers            : 12,500
accounts imputed (0->1): 475

accounts per customer:
count    12500.000000
mean         5.505520
std          2.446002
min          1.000000
25%          4.000000
50%          6.000000
75%          7.000000
max         11.000000

split sizes:
Split
train     7509
val       1891
test      1926
system    1174


## Step 3: Accounts and persistent counterparties

Each customer opens `Num_Bank_Accounts` accounts at real US bank IDs drawn from the reference
data. Every customer then gets a **persistent** counterparty set they reuse all period:

* **1 employer** recurring inbound salary
* **5–15 merchants** from a shared 40,000-account pool, so merchants are shared across
  customers and become high-degree nodes
* **3–8 peers** accounts belonging to *other simulated customers*, which is what connects
  customers to each other and makes cycles possible

Counterparty choice within a customer's set is Zipf-weighted, so a few counterparties dominate as in real spending. This is the step the old version omitted entirely, and without it every
graph feature is zero regardless of what the model does.

In [4]:
# --- mint customer accounts --------------------------------------------------
acct_counts = profile["Num_Bank_Accounts"].to_numpy()
total_accounts = int(acct_counts.sum())

customer_of_account = np.repeat(profile["Customer_ID"].to_numpy(), acct_counts)
account_ids = np.array([f"ACC{i:08d}" for i in range(total_accounts)])
account_banks = rng.choice(US_BANK_IDS, size=total_accounts)

# offset[i] : index of customer i's first account in the flat arrays above
account_offset = np.concatenate([[0], np.cumsum(acct_counts)])

# --- shared external pool ----------------------------------------------------
merchant_ids = np.array([f"MER{i:07d}" for i in range(N_MERCHANT_ACCOUNTS)])
merchant_banks = rng.choice(US_BANK_IDS, size=N_MERCHANT_ACCOUNTS)
employer_ids = np.array([f"EMP{i:06d}" for i in range(2_000)])
employer_banks = rng.choice(US_BANK_IDS, size=2_000)

def zipf_weights(k: int) -> np.ndarray:
    w = 1.0 / np.power(np.arange(1, k + 1), ZIPF_A)
    return w / w.sum()

# --- per-customer counterparty sets -----------------------------------------
cust_merchants, cust_peers, cust_employer = [], [], []
for i in range(N_CUSTOMERS):
    k_m = rng.integers(*MERCHANTS_PER_CUSTOMER)
    cust_merchants.append(rng.choice(N_MERCHANT_ACCOUNTS, size=k_m, replace=False))
    k_p = rng.integers(*PEERS_PER_CUSTOMER)
    # peers are other customers' accounts -> inter-customer edges
    cust_peers.append(rng.integers(0, total_accounts, size=k_p))
    cust_employer.append(rng.integers(0, len(employer_ids)))

print(f"customer accounts    : {total_accounts:,}")
print(f"merchant pool        : {N_MERCHANT_ACCOUNTS:,}")
print(f"mean merchants/cust  : {np.mean([len(m) for m in cust_merchants]):.1f}")
print(f"mean peers/cust      : {np.mean([len(p) for p in cust_peers]):.1f}")

customer accounts    : 68,819
merchant pool        : 40,000
mean merchants/cust  : 9.5
mean peers/cust      : 5.0


## Step 4 — Legitimate transactions

Two streams, per the volume model:

* **External** : `Poisson(λᵢ × 6)` per customer, *independent of account count*, spread across
  their accounts. Salary inbound from the employer; the rest outbound to merchants and peers.
* **Internal** : for each of the `n − 1` adjacent own-account routes,
  `Poisson(3 × activityᵢ × 6)` transfers. This is the only stream that scales with
  `Num_Bank_Accounts`.

### Where the rate λᵢ comes from

```
λᵢ = 47  ×  occupation_multiplier  ×  activity_factorᵢ
     ^^^     ^^^^^^^^^^^^^^^^^^^^     ^^^^^^^^^^^^^^^^
     DCPC    reasoned tier            Gamma(4, ¼), mean 1.0
```

Drawing each customer's rate from a Gamma and *then* sampling Poisson gives a negative
binomial marginally, which decouples variance from mean. Same average, realistic spread.

**Why the occupation tiers are labelled assumptions.** The four tiers encode mechanisms (self-employment mixing
business and personal flows; gig income arriving from many payers; cash-intensive trades)
rather than measurements, and the constant block says so. Only the 47/month baseline is cited.

In [5]:
# Hour-of-day weights: retail activity concentrates 08:00-21:00.
HOUR_W = np.array([0.4,0.3,0.2,0.2,0.3,0.6,1.2,2.2,3.6,4.6,5.2,5.6,
                   5.8,5.6,5.4,5.2,5.4,5.8,5.6,4.8,3.8,2.8,1.8,0.9])
HOUR_W = HOUR_W / HOUR_W.sum()
DOW_W = np.array([1.10, 1.08, 1.06, 1.08, 1.15, 0.80, 0.63])   # Mon..Sun
DOW_W = DOW_W / DOW_W.sum()

def sample_timestamps(k, generator, day_lo=0, day_hi=SIMULATION_DAYS):
    """Business-hours and weekday weighted timestamps within [day_lo, day_hi)."""
    days = np.arange(day_lo, day_hi)
    dow = (START_DATE.dayofweek + days) % 7
    p = DOW_W[dow]; p = p / p.sum()
    chosen_days = generator.choice(days, size=k, p=p)
    hours = generator.choice(24, size=k, p=HOUR_W)
    minutes = generator.integers(0, 60, size=k)
    return (START_DATE
            + pd.to_timedelta(chosen_days, unit="D")
            + pd.to_timedelta(hours, unit="h")
            + pd.to_timedelta(minutes, unit="m"))

def sample_amounts(formats, generator):
    """Amount conditional on payment format, from the Step 1 fit."""
    out = np.empty(len(formats))
    for fmt in np.unique(formats):
        mask = formats == fmt
        m = AMOUNT_MODELS[fmt]
        out[mask] = generator.lognormal(m["mean_log"], m["std_log"], size=int(mask.sum()))
    return np.round(out, 2)

FORMAT_PROBS = FORMAT_P.to_numpy()
FORMAT_ARR = np.array(FORMATS)

# --- per-customer rates, drawn once so they can be recorded on the profile ---
# occupation multiplier: a fixed tier (see OCCUPATION_RATE_MULTIPLIER)
# activity factor: this individual's own propensity to transact, mean 1.0
# The activity factor multiplies BOTH streams. Someone who transacts a lot does
# so externally and between their own accounts; splitting the two would imply a
# person can be simultaneously busy and idle.
occ_multiplier = (profile["Occupation"].map(OCCUPATION_RATE_MULTIPLIER)
                  .fillna(DEFAULT_RATE_MULTIPLIER).to_numpy())
activity_factor = rng.gamma(GAMMA_SHAPE, 1.0 / GAMMA_SHAPE, size=N_CUSTOMERS)

profile["Rate_Multiplier"] = occ_multiplier
profile["Activity_Factor"] = np.round(activity_factor, 4)

print("expected external payments/month by occupation tier:")
for mult in sorted(set(occ_multiplier), reverse=True):
    jobs = sorted(profile.loc[occ_multiplier == mult, "Occupation"].unique())
    print(f"  {mult:.2f}x -> {EXT_PAYMENTS_PER_MONTH*mult:5.1f}/month   "
          f"{', '.join(jobs[:5])}{' ...' if len(jobs) > 5 else ''}")
print()

blocks = []
_t0 = time.time()
print(f"generating legitimate transactions for {N_CUSTOMERS:,} customers")
print("  (the ETA stabilises after the first few hundred)")
for i in range(N_CUSTOMERS):
    progress(i + 1, N_CUSTOMERS, _t0, "customers", every=500)
    cid = profile["Customer_ID"].iat[i]
    n_acc = int(acct_counts[i])
    my_accounts = np.arange(account_offset[i], account_offset[i + 1])

    # ---------------- external payments (volume independent of n_acc) --------
    # Gamma-mixed Poisson: this customer's own rate, then a draw given that rate.
    lam_ext = EXT_PAYMENTS_PER_MONTH * occ_multiplier[i] * activity_factor[i]
    n_ext = int(rng.poisson(lam_ext * MONTHS))
    n_salary = int(rng.poisson(2.0 * MONTHS))          # ~2 pay runs per month
    n_ext = max(n_ext, n_salary + 1)
    n_out = n_ext - n_salary

    merchants = cust_merchants[i]
    peers = cust_peers[i]
    pool = np.concatenate([merchants, peers + N_MERCHANT_ACCOUNTS])   # tagged pool
    picks = rng.choice(len(pool), size=n_out, p=zipf_weights(len(pool)))
    chosen = pool[picks]
    is_merchant = chosen < N_MERCHANT_ACCOUNTS

    cp_acct = np.empty(n_out, dtype=object)
    cp_bank = np.empty(n_out, dtype=np.int64)
    cp_acct[is_merchant] = merchant_ids[chosen[is_merchant]]
    cp_bank[is_merchant] = merchant_banks[chosen[is_merchant]]
    peer_idx = chosen[~is_merchant] - N_MERCHANT_ACCOUNTS
    cp_acct[~is_merchant] = account_ids[peer_idx]
    cp_bank[~is_merchant] = account_banks[peer_idx]

    own_out = rng.choice(my_accounts, size=n_out)
    fmt_out = rng.choice(FORMAT_ARR, size=n_out, p=FORMAT_PROBS)

    emp = cust_employer[i]
    own_in = rng.choice(my_accounts, size=n_salary)
    fmt_in = np.full(n_salary, "ACH")

    ext = pd.DataFrame({
        "Timestamp": np.concatenate([sample_timestamps(n_out, rng).to_numpy(),
                                     sample_timestamps(n_salary, rng).to_numpy()]),
        "From Bank": np.concatenate([account_banks[own_out], np.full(n_salary, employer_banks[emp])]),
        "Account":   np.concatenate([account_ids[own_out], np.full(n_salary, employer_ids[emp])]),
        "To Bank":   np.concatenate([cp_bank, account_banks[own_in]]),
        "Account.1": np.concatenate([cp_acct, account_ids[own_in]]),
        "Payment Format": np.concatenate([fmt_out, fmt_in]),
        "_role": np.concatenate([np.full(n_out, "sender"), np.full(n_salary, "receiver")]),
        "_stream": "external",
    })

    # ---------------- internal transfers (scale with n_acc) ------------------
    if n_acc > 1:
        parts = []
        for r in range(n_acc - 1):
            k = int(rng.poisson(TRANSFERS_PER_ROUTE_MONTH * activity_factor[i] * MONTHS))
            if k == 0:
                continue
            src, dst = my_accounts[r], my_accounts[r + 1]
            parts.append(pd.DataFrame({
                "Timestamp": sample_timestamps(k, rng).to_numpy(),
                "From Bank": account_banks[src], "Account": account_ids[src],
                "To Bank": account_banks[dst],   "Account.1": account_ids[dst],
                "Payment Format": rng.choice(["ACH", "Cheque"], size=k, p=[0.8, 0.2]),
                "_role": "sender", "_stream": "internal",
            }))
        if parts:
            ext = pd.concat([ext] + parts, ignore_index=True)

    ext["Customer_ID"] = cid
    blocks.append(ext)

_t = stage(f"concatenating {len(blocks):,} per-customer frames")
legit = pd.concat(blocks, ignore_index=True)
# Release the per-customer frames immediately. Holding both the list and the
# concatenated result doubles peak memory at the worst moment, and at 12,500
# customers that is a couple of GB that can push a 8-16 GB machine into swap.
del blocks
stage("concatenate", _t)

_t = stage(f"sampling amounts for {len(legit):,} rows")
legit["Amount Received"] = sample_amounts(legit["Payment Format"].to_numpy(), rng)
stage("amount sampling", _t)
legit["Is Laundering"] = 0
legit["Typology"] = "NONE"

print(f"legitimate transactions : {len(legit):,}")
print(f"per customer (mean)     : {len(legit)/N_CUSTOMERS:.1f}")
print(f"stream mix              :\n{legit['_stream'].value_counts().to_string()}")

expected external payments/month by occupation tier:
  1.60x ->  75.2/month   Entrepreneur
  1.20x ->  56.4/month   Journalist, Musician, Writer
  1.15x ->  54.0/month   Mechanic
  1.00x ->  47.0/month   Unknown
  0.95x ->  44.6/month   Accountant, Architect, Developer, Doctor, Engineer ...

generating legitimate transactions for 12,500 customers
  (the ETA stabilises after the first few hundred)
  customers: 500/12,500 (  4.0%)  elapsed 6s  eta 2m46s
  customers: 1,000/12,500 (  8.0%)  elapsed 13s  eta 2m40s
  customers: 1,500/12,500 ( 12.0%)  elapsed 20s  eta 2m32s
  customers: 2,000/12,500 ( 16.0%)  elapsed 27s  eta 2m25s
  customers: 2,500/12,500 ( 20.0%)  elapsed 34s  eta 2m16s
  customers: 3,000/12,500 ( 24.0%)  elapsed 42s  eta 2m13s
  customers: 3,500/12,500 ( 28.0%)  elapsed 49s  eta 2m06s
  customers: 4,000/12,500 ( 32.0%)  elapsed 55s  eta 1m58s
  customers: 4,500/12,500 ( 36.0%)  elapsed 1m02s  eta 1m51s
  customers: 5,000/12,500 ( 40.0%)  elapsed 1m08s  eta 1m43s
  custome

## Step 5: Typology injection

This is where the label comes from. 3% of customers are selected as laundering participants;
each is assigned one typology, and **every transaction generated by that structure is labelled
1**. No other transaction is ever labelled.

| Typology | Structure | What makes it detectable |
|---|---|---|
| `STRUCTURING` | 4–9 deposits of \$2,000–\$9,900 into one account within 3–10 days | Amount pile-up just under the \$10,000 CTR threshold, plus a velocity burst |
| `FAN_IN` | 6–15 peer accounts → one account within 7 days | In-degree spike on the receiving account |
| `FAN_OUT` | One account → 6–15 accounts within 7 days | Out-degree spike |
| `SCATTER_GATHER` | Source → k intermediates → single destination | 2-hop pattern; invisible to per-transaction features |
| `CYCLE` | A → B → C → A within a window | Funds return to origin; needs 3-hop traversal |

Formats are drawn from the **laundering-conditional** mix measured in Step 1 (ACH-dominated in
the real data), not the general mix. That reproduces an observed property of the reference data
rather than inventing one.

The last three typologies are precisely why graph features matter: no amount of per-transaction
feature engineering will surface a cycle.

In [6]:
rng_l = np.random.default_rng(SEED + 7)

n_laundering = int(round(N_CUSTOMERS * LAUNDERING_CUSTOMER_RATE))
laundering_idx = rng_l.choice(N_CUSTOMERS, size=n_laundering, replace=False)
assigned = rng_l.choice(TYPOLOGIES, size=n_laundering, p=TYPOLOGY_WEIGHTS)

LFMT = FORMAT_P_LAUNDERING.to_numpy()

def _rows(ts, src, dst, fmt, amt, cid, typ):
    return pd.DataFrame({
        "Timestamp": ts,
        "From Bank": account_banks[src], "Account": account_ids[src],
        "To Bank": account_banks[dst], "Account.1": account_ids[dst],
        "Payment Format": fmt, "Amount Received": np.round(amt, 2),
        "_role": "sender", "_stream": "laundering",
        "Customer_ID": cid, "Is Laundering": 1, "Typology": typ,
    })

laundering_blocks = []
_t0 = time.time()
print(f"injecting typologies into {n_laundering:,} customers")
for k, i in enumerate(laundering_idx):
    progress(k + 1, n_laundering, _t0, "laundering customers", every=100)
    cid = profile["Customer_ID"].iat[i]
    typ = assigned[k]
    my_accounts = np.arange(account_offset[i], account_offset[i + 1])
    peers = cust_peers[i]
    start_day = int(rng_l.integers(0, SIMULATION_DAYS - 14))

    if typ == "STRUCTURING":
        n = int(rng_l.integers(4, 10))
        target = my_accounts[rng_l.integers(0, len(my_accounts))]
        src = rng_l.choice(peers, size=n)
        amt = np.clip(STRUCTURING_MAX - rng_l.exponential(STRUCTURING_DECAY, size=n),
                      STRUCTURING_MIN, STRUCTURING_MAX)
        ts = sample_timestamps(n, rng_l, start_day, start_day + int(rng_l.integers(3, 11)))
        fmt = rng_l.choice(FORMAT_ARR, size=n, p=LFMT)
        laundering_blocks.append(_rows(ts.to_numpy(), src, np.full(n, target), fmt, amt, cid, typ))

    elif typ == "FAN_IN":
        n = int(rng_l.integers(6, 16))
        target = my_accounts[rng_l.integers(0, len(my_accounts))]
        src = rng_l.integers(0, total_accounts, size=n)
        amt = rng_l.lognormal(AMOUNT_MODELS["ACH"]["mean_log"], 1.2, size=n)
        ts = sample_timestamps(n, rng_l, start_day, start_day + 7)
        fmt = rng_l.choice(FORMAT_ARR, size=n, p=LFMT)
        laundering_blocks.append(_rows(ts.to_numpy(), src, np.full(n, target), fmt, amt, cid, typ))

    elif typ == "FAN_OUT":
        n = int(rng_l.integers(6, 16))
        source = my_accounts[rng_l.integers(0, len(my_accounts))]
        dst = rng_l.integers(0, total_accounts, size=n)
        amt = rng_l.lognormal(AMOUNT_MODELS["ACH"]["mean_log"], 1.2, size=n)
        ts = sample_timestamps(n, rng_l, start_day, start_day + 7)
        fmt = rng_l.choice(FORMAT_ARR, size=n, p=LFMT)
        laundering_blocks.append(_rows(ts.to_numpy(), np.full(n, source), dst, fmt, amt, cid, typ))

    elif typ == "SCATTER_GATHER":
        k_mid = int(rng_l.integers(3, 8))
        source = my_accounts[rng_l.integers(0, len(my_accounts))]
        mids = rng_l.choice(peers, size=k_mid) if len(peers) >= k_mid else rng_l.integers(0, total_accounts, size=k_mid)
        final = rng_l.integers(0, total_accounts)
        amt = rng_l.lognormal(AMOUNT_MODELS["ACH"]["mean_log"], 1.0, size=k_mid)
        t1 = sample_timestamps(k_mid, rng_l, start_day, start_day + 3)
        t2 = sample_timestamps(k_mid, rng_l, start_day + 3, start_day + 8)
        f1 = rng_l.choice(FORMAT_ARR, size=k_mid, p=LFMT)
        f2 = rng_l.choice(FORMAT_ARR, size=k_mid, p=LFMT)
        laundering_blocks.append(_rows(t1.to_numpy(), np.full(k_mid, source), mids, f1, amt, cid, typ))
        laundering_blocks.append(_rows(t2.to_numpy(), mids, np.full(k_mid, final), f2, amt * 0.97, cid, typ))

    elif typ == "CYCLE":
        a = my_accounts[rng_l.integers(0, len(my_accounts))]
        b, c = rng_l.integers(0, total_accounts, size=2)
        amt0 = float(rng_l.lognormal(AMOUNT_MODELS["ACH"]["mean_log"], 1.0))
        ts = np.sort(sample_timestamps(3, rng_l, start_day, start_day + 10).to_numpy())
        fmt = rng_l.choice(FORMAT_ARR, size=3, p=LFMT)
        laundering_blocks.append(_rows(ts, np.array([a, b, c]), np.array([b, c, a]),
                                       fmt, np.array([amt0, amt0*0.98, amt0*0.96]), cid, typ))

laundering = pd.concat(laundering_blocks, ignore_index=True)

print(f"laundering customers   : {n_laundering:,}  ({100*n_laundering/N_CUSTOMERS:.2f}% of customers)")
print(f"laundering transactions: {len(laundering):,}")
print(f"per customer (mean)    : {len(laundering)/n_laundering:.1f}")
print(f"\ntypology mix:\n{pd.Series(assigned).value_counts().to_string()}")

injecting typologies into 375 customers
  laundering customers: 100/375 ( 26.7%)  elapsed 0s  eta 0s
  laundering customers: 200/375 ( 53.3%)  elapsed 0s  eta 0s
  laundering customers: 300/375 ( 80.0%)  elapsed 0s  eta 0s
  laundering customers: 375/375 (100.0%)  elapsed 0s  eta 0s
laundering customers   : 375  (3.00% of customers)
laundering transactions: 3,061
per customer (mean)    : 8.2

typology mix:
STRUCTURING       110
FAN_IN             79
FAN_OUT            79
SCATTER_GATHER     54
CYCLE              53


## Step 6: Assemble, split, write

Columns match the `transactions` table in `database/customer_db.py`, so
`load_transactions_from_csv` keeps working unchanged. `Split` is carried on each row so the
four-way partition needs no recomputation downstream and it comes from the same MD5 buckets
`ml_models/data_splits.py` uses, so AML and CDA share one customer partition.

In [7]:
_t = stage("merging legitimate and laundering transactions")
txns = pd.concat([legit, laundering], ignore_index=True)
# Frees ~2 GB before the sort and the CSV write, the two peak-memory steps.
# Note this makes cells 4 -> 6 order-dependent: re-running this cell alone will
# raise NameError, so re-run the generation cell first. "Run All" is unaffected.
del legit
stage("merge", _t)

# Single-currency, US-only by construction.
txns["Receiving Currency"] = "US Dollar"
txns["Payment Currency"] = "US Dollar"
txns["Amount Paid"] = txns["Amount Received"]

_t = stage(f"sorting {len(txns):,} rows by timestamp")
txns = txns.sort_values("Timestamp", kind="mergesort").reset_index(drop=True)
stage("sort", _t)

_t = stage("assigning transaction IDs and split labels")
txns["Transaction_ID"] = [f"TXN{i:09d}" for i in range(len(txns))]
txns["Split"] = txns["Customer_ID"].map(profile.set_index("Customer_ID")["Split"])
stage("IDs and splits", _t)

COLUMNS = ["Transaction_ID", "Timestamp", "From Bank", "Account", "To Bank", "Account.1",
           "Amount Received", "Receiving Currency", "Amount Paid", "Payment Currency",
           "Payment Format", "Is Laundering", "Typology", "_role", "_stream",
           "Customer_ID", "Split"]
txns = txns[COLUMNS]

profile_out = profile.merge(
    txns.groupby("Customer_ID").agg(N_Transactions=("Transaction_ID", "size"),
                                    N_Laundering=("Is Laundering", "sum")).reset_index(),
    on="Customer_ID", how="left")

# The single slowest step at full scale: pandas formats every field as text,
# single-threaded, for an ~800 MB file. Low CPU with steady disk writes here is
# normal, not a stall.
_t = stage(f"writing {len(txns):,} rows to customer_transactions_train.csv (~800 MB)")
txns.to_csv(OUT_DIR / "customer_transactions_train.csv", index=False)
stage("transactions CSV", _t)

_t = stage("writing customer_profile_train.csv")
profile_out.to_csv(OUT_DIR / "customer_profile_train.csv", index=False)
stage("profile CSV", _t)

print()
print(f"rows written : {len(txns):,}")
print(f"laundering   : {int(txns['Is Laundering'].sum()):,} "
      f"({100*txns['Is Laundering'].mean():.4f}% of transactions)")
print(f"date range   : {txns['Timestamp'].min()} .. {txns['Timestamp'].max()}")
print(f"\nrows per split:\n{txns.groupby('Split')['Is Laundering'].agg(['size','sum','mean']).to_string()}")
print(f"\nwritten to {OUT_DIR}")

  merging legitimate and laundering transactions ...
  merge done in 0s
  sorting 4,705,360 rows by timestamp ...
  sort done in 11s
  assigning transaction IDs and split labels ...
  IDs and splits done in 4s
  writing 4,705,360 rows to customer_transactions_train.csv (~800 MB) ...
  transactions CSV done in 1m00s
  writing customer_profile_train.csv ...
  profile CSV done in 0s

rows written : 4,705,360
laundering   : 3,061 (0.0651% of transactions)
date range   : 2024-01-01 00:00:00 .. 2024-06-30 23:59:00

rows per split:
           size   sum      mean
Split                          
system   450354   266  0.000591
test     732025   563  0.000769
train   2802216  1801  0.000643
val      720765   431  0.000598

written to C:\dev\FRAML_PROJECT\data\simulated


## Step 7: Validation

Assertions rather than printed summaries. A validation cell that prints numbers can look
thorough and passed while the label was pure noise, it checked an identity (`Estimated_Annual_Income`
against `Annual_Income`) that the generator had just forced to hold. These check the properties
that were actually broken, and a failure stops the notebook.

In [8]:
fails = []
def check(name, condition, detail=""):
    if condition:
        print(f"  PASS  {name}")
    else:
        fails.append(f"{name} :: {detail}")
        print(f"  FAIL  {name}  -- {detail}")

# The CSVs are already on disk by this point -- these checks read the in-memory
# frame and write nothing. Interrupting here loses the report, not the data.
print(f"validating {len(txns):,} rows. Each group below scans the full table, "
      f"so allow a minute or two at full scale.\n")

print("LABEL INTEGRITY")
check("label comes only from typologies",
      (txns.loc[txns["Is Laundering"] == 1, "Typology"] != "NONE").all(),
      "some labelled rows have no typology")
check("no unlabelled typology rows",
      (txns.loc[txns["Typology"] != "NONE", "Is Laundering"] == 1).all())

print("\nLABEL IS LEARNABLE (these all failed in the old version)")
by_fmt = txns.groupby("Payment Format")["Is Laundering"].mean()
check("format separates label", by_fmt.max() > 3 * by_fmt.min(),
      f"max/min rate ratio only {by_fmt.max()/max(by_fmt.min(),1e-12):.2f}")

struct = txns[txns["Typology"] == "STRUCTURING"]
band = txns[(txns["Amount Received"] >= 9000) & (txns["Amount Received"] < 10000)]
check("structuring lands under the CTR threshold",
      struct["Amount Received"].between(STRUCTURING_MIN, STRUCTURING_MAX).all())
check("a sub-$10k pile-up exists", band["Is Laundering"].mean() > txns["Is Laundering"].mean() * 3,
      f"laundering rate in $9-10k band {band['Is Laundering'].mean():.5f} vs base {txns['Is Laundering'].mean():.5f}")

print("\nGRAPH STRUCTURE (all-zero in the old version)")
deg = txns.groupby("Account.1").size()
check("counterparties are reused", deg.mean() > 2.0, f"mean in-degree only {deg.mean():.2f}")
check("high-degree hubs exist", deg.max() > 50, f"max in-degree {deg.max()}")
peers_seen = txns[txns["Account.1"].str.startswith("ACC")]
check("customer-to-customer edges exist", len(peers_seen) > 0)

print("\nAMOUNT REALISM")
share_10k = (txns["Amount Received"] >= 10000).mean()
check("meaningful mass above $10k", share_10k > 0.05, f"only {100*share_10k:.2f}% >= $10k")

print("\nSPLIT INTEGRITY")
per_split = txns.groupby("Customer_ID")["Split"].nunique()
check("no customer spans two splits", (per_split == 1).all())
check("every split has positives", (txns.groupby("Split")["Is Laundering"].sum() > 0).all())

print("\nWINDOW")
span = (txns["Timestamp"].max() - txns["Timestamp"].min()).days
check("~6 months of data", 150 <= span <= 190, f"span {span} days")

print("\nVOLUME MODEL (occupation tiers + Gamma-mixed Poisson)")
per_cust = txns.groupby("Customer_ID", observed=True).size()
dispersion = per_cust.var() / per_cust.mean()
check("counts are overdispersed vs plain Poisson", dispersion > 2.0,
      f"dispersion index {dispersion:.2f}; plain Poisson would give ~1.0 within a tier")

by_occ = profile_out.groupby("Occupation")["N_Transactions"].mean()
check("entrepreneurs transact more than salaried professionals",
      by_occ.get("Entrepreneur", 0) > by_occ.get("Teacher", 0) * 1.3,
      f"Entrepreneur {by_occ.get('Entrepreneur', 0):.0f} vs Teacher {by_occ.get('Teacher', 0):.0f}")

# Within one occupation and account count, customers should still differ. If they
# do not, the Gamma mix is not being applied and we are back to plain Poisson.
same_tier = profile_out[(profile_out["Occupation"] == "Teacher")
                        & (profile_out["Num_Bank_Accounts"] == 5)]["N_Transactions"]
if len(same_tier) > 20:
    cv = same_tier.std() / same_tier.mean()
    check("customers within one tier genuinely differ", cv > 0.25,
          f"coefficient of variation only {cv:.3f} among {len(same_tier)} "
          f"Teachers with 5 accounts")

if fails:
    raise AssertionError("Validation failed:\n  - " + "\n  - ".join(fails))
print("\nAll validation checks passed.")

validating 4,705,360 rows. Each group below scans the full table, so allow a minute or two at full scale.

LABEL INTEGRITY
  PASS  label comes only from typologies
  PASS  no unlabelled typology rows

LABEL IS LEARNABLE (these all failed in the old version)
  PASS  format separates label
  PASS  structuring lands under the CTR threshold
  PASS  a sub-$10k pile-up exists

GRAPH STRUCTURE (all-zero in the old version)
  PASS  counterparties are reused
  PASS  high-degree hubs exist
  PASS  customer-to-customer edges exist

AMOUNT REALISM
  PASS  meaningful mass above $10k

SPLIT INTEGRITY
  PASS  no customer spans two splits
  PASS  every split has positives

WINDOW
  PASS  ~6 months of data

VOLUME MODEL (occupation tiers + Gamma-mixed Poisson)
  PASS  counts are overdispersed vs plain Poisson
  PASS  entrepreneurs transact more than salaried professionals
  PASS  customers within one tier genuinely differ

All validation checks passed.


## Step 8 — Diagnostics

Laundering rate by hour and
by split *should* stay flat — those are not meant to carry signal. Format, amount and degree
*should* separate sharply. If any of the three is flat, the injection did not work.

In [9]:
print("computing diagnostics (several full-table group-bys; nothing is written)\n")
base = txns["Is Laundering"].mean()
print(f"base laundering rate: {100*base:.4f}%\n")

print("laundering rate by payment format (should SEPARATE):")
t = txns.groupby("Payment Format")["Is Laundering"].agg(["size", "mean"])
for f, r in t.sort_values("mean", ascending=False).iterrows():
    print(f"  {f:<14} n={int(r['size']):>9,}  rate={100*r['mean']:>8.4f}%  ({r['mean']/base:>5.1f}x base)")

print("\nlaundering rate by amount band (should SPIKE just under $10k):")
bands = [(0,1000),(1000,5000),(5000,9000),(9000,10000),(10000,50000),(50000,1e12)]
for lo, hi in bands:
    m = (txns["Amount Received"] >= lo) & (txns["Amount Received"] < hi)
    if m.sum():
        print(f"  ${lo:>7,.0f}-${hi:>10,.0f}  n={int(m.sum()):>9,}  "
              f"rate={100*txns.loc[m,'Is Laundering'].mean():>8.4f}%  "
              f"({txns.loc[m,'Is Laundering'].mean()/base:>5.1f}x)")

print("\nlaundering rate by hour (should stay FLAT -- hour is not a real signal):")
h = txns.groupby(txns["Timestamp"].dt.hour)["Is Laundering"].agg(["size", "sum", "mean"])
print(f"  rate range {100*h['mean'].min():.4f}% .. {100*h['mean'].max():.4f}%")
print(f"  positives per hour: min {int(h['sum'].min())}, max {int(h['sum'].max())}, "
      f"median {int(h['sum'].median())}")
print("  Read counts, not the ratio: quiet overnight hours hold few positives, so the")
print("  ratio of rates is dominated by small-sample noise and can even divide by zero.")

print("\ncounterparty in-degree (0 patterns were possible in the old data):")
deg = txns.groupby("Account.1").size()
print(f"  mean {deg.mean():.2f} | median {deg.median():.0f} | p99 {deg.quantile(0.99):.0f} | max {deg.max()}")

print("\ntransactions per customer by occupation (6 months):")
occ_stats = (profile_out.groupby("Occupation")
             .agg(customers=("Customer_ID", "size"),
                  mult=("Rate_Multiplier", "first"),
                  mean_txns=("N_Transactions", "mean"),
                  std_txns=("N_Transactions", "std"))
             .sort_values("mean_txns", ascending=False))
occ_stats["cv"] = occ_stats["std_txns"] / occ_stats["mean_txns"]
print(f"  {'occupation':<16}{'n':>6}{'mult':>7}{'mean':>9}{'std':>8}{'cv':>7}")
for occ, r in occ_stats.iterrows():
    print(f"  {occ:<16}{int(r['customers']):>6}{r['mult']:>7.2f}"
          f"{r['mean_txns']:>9.0f}{r['std_txns']:>8.0f}{r['cv']:>7.2f}")

per_cust = txns.groupby("Customer_ID", observed=True).size()
print(f"\n  population dispersion index (var/mean): {per_cust.var()/per_cust.mean():.2f}")
print("  Plain Poisson gives ~1.0 within a homogeneous tier. Anything well above")
print("  that is the Gamma mixture doing its job.")

print("\ntypology breakdown:")
print(txns[txns["Typology"] != "NONE"].groupby("Typology")
      .agg(transactions=("Transaction_ID", "size"),
           customers=("Customer_ID", "nunique"),
           median_amount=("Amount Received", "median")).to_string())

computing diagnostics (several full-table group-bys; nothing is written)

base laundering rate: 0.0655%

laundering rate by payment format (should SEPARATE):
  ACH            n=1,323,016  rate=  0.1751%  (  2.7x base)
  Cash           n=  353,354  rate=  0.0300%  (  0.5x base)
  Credit Card    n=  967,916  rate=  0.0258%  (  0.4x base)
  Cheque         n=1,550,502  rate=  0.0254%  (  0.4x base)
  Reinvestment   n=  363,320  rate=  0.0000%  (  0.0x base)
  Wire           n=  122,030  rate=  0.0000%  (  0.0x base)

laundering rate by amount band (should SPIKE just under $10k):
  $      0-$     1,000  n=2,319,829  rate=  0.0460%  (  0.7x)
  $  1,000-$     5,000  n=1,015,681  rate=  0.1083%  (  1.7x)
  $  5,000-$     9,000  n=  305,792  rate=  0.1665%  (  2.5x)
  $  9,000-$    10,000  n=   50,515  rate=  0.6493%  (  9.9x)
  $ 10,000-$    50,000  n=  571,191  rate=  0.0109%  (  0.2x)
  $ 50,000-$1,000,000,000,000  n=  417,130  rate=  0.0002%  (  0.0x)

laundering rate by hour (should stay F